# Учимся определять ценовой класс квартиры

Теперь попробуем классификацию: будем прогнозировать класс квартиры по её характеристикам и общей цене. Готовая метка записана в `above_reference`; её смысл мы обсуждали на занятии.

Здесь пригодится материал из папки `lesson_4` репозитория курса. В местах `TODO` вы сможете собрать обработку, выбрать модель, а затем проверить результат. Пока подготовка данных и модель не дописаны, остановка кода ожидаема — подсказка покажет, к какому шагу вернуться. Общее задание находится в файле `TASK.md` в корне папки `final_project` — оно поможет держать весь проект в поле зрения.

Общая цена квартиры `price_rub` в рублях входит в признаки вместе с площадью и другими характеристиками. При использовании приложения пользователь также указывает цену.

<details>
<summary>Если хочется вспомнить математику</summary>

Логистическая регрессия получает вероятность так: $p=1/(1+e^{-z})$, где $z=b+\sum_j w_jx_j$.

Чтобы перейти к классу, выбираем порог $t$: $\hat y=1$, если $p\ge t$, иначе 0. В этом проекте порог вероятности равен 0.5.

$F1=2PR/(P+R)$, где $P$ — precision, $R$ — recall. Посмотрите на обе метрики отдельно: они помогут понять, какие ошибки делает модель.

</details>


### Как найти места для вашей работы

В ноутбуке есть пометки `TODO` — в коде они выглядят как `# TODO`. Попробуйте поиск по слову `TODO` через `Ctrl+F` (`Cmd+F` на macOS): так будет проще пройти все шаги и ничего не упустить. Перед завершением ещё раз просмотрите найденные места.

Код дописывайте в кодовых ячейках, а наблюдения и объяснения — в Markdown под строкой `Ответ:`.

В ноутбуке используются `assert` — это готовые проверки, которые помогают убедиться, что данные и результаты имеют ожидаемый вид. В самих проверках ничего дописывать не нужно: просто запускайте их вместе с остальными ячейками. Если проверка сообщает об ошибке, вернитесь к соответствующему шагу и проверьте свой код.

In [ ]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data").is_dir(), "Откройте ноутбук из папки final_project."

In [ ]:
df = pd.read_csv(ROOT / "data" / "classification.csv")
TARGET = "above_reference"
SERVICE = ["listing_id", "group_id", "split"]
df.head()

## 1. Познакомимся с данными

Начнём со знакомых действий из анализа данных. Посмотрите на типы столбцов, пропуски и распределения, а затем запишите 2–3 наблюдения, которые показались вам интересными.

Здесь пригодятся `info()`, `isna().mean()`, `describe()`, `value_counts()`, `hist()` и `scatter()`. Примеры можно найти в папке `lesson_2` репозитория курса. Заполнение пропусков пока оставим для обработки внутри модели — её мы будем учить только на train.

Посмотрите отдельно на баланс классов: одинаково ли часто встречаются 0 и 1?


In [ ]:
# TODO: добавьте свой EDA и несколько понятных графиков.
df.info()
df.isna().mean().sort_values(ascending=False)

**TODO:** Запишите 2–3 наблюдения по анализу данных. Какие особенности вы учтёте при подготовке модели?

Ответ:

## 2. Разделим данные на train и test

Для простоты используем две части: на `train` учим обработку и модель, на `test` проверяем результат. Разбиение уже подготовлено в `split`. Сохраним его: связанные объявления должны оставаться в одной части.

In [ ]:
assert set(df["split"]) == {"train", "test"}
assert df.groupby("group_id")["split"].nunique().max() == 1
train = df.loc[df["split"].eq("train")].copy()
test = df.loc[df["split"].eq("test")].copy()
assert not train.empty and not test.empty
assert set(train["group_id"]).isdisjoint(test["group_id"])
print({"train": len(train), "test": len(test)})

## 3. Подготовим признаки простыми шагами

Вспомните, как мы отдельно работали с числами и категориями. Здесь можно использовать знакомые `fillna()` и `pd.get_dummies()`. Это подсказки, а не обязательный рецепт: свои идеи тоже приветствуются.

В `prepare_features` допишите подготовку таблицы. При первом вызове на train сохраните нужные значения в словарь `preparation`: например, медианы и названия столбцов после кодирования. При вызове на test используйте тот же словарь. Новые категории не должны менять список столбцов: здесь поможет `reindex(..., fill_value=0)`.

Масштабирование необязательно. Если его используете, обучите scaler только на train и также сохраните его в `preparation`. Для приложения повторите те же шаги обработки с сохранёнными значениями. Описание полей находится в `README.md` в папке `final_project/data`, а примеры — в папках `lesson_2`, `lesson_3` и `lesson_4` репозитория курса.

In [ ]:
features = [
    "area", "rooms", "floor", "floors_total", "lift_count",
    "city", "house_type", "market_type", "price_rub",
]
assert features and len(features) == len(set(features))
assert set(features).issubset(df.columns)
forbidden = set(SERVICE + [TARGET])
assert not set(features) & forbidden, "Исключите цели и служебные поля."
assert "price_rub" in features, "Для классификации добавьте общую цену квартиры."

In [ ]:
def prepare_features(frame, preparation=None):
    data = frame.copy()
    # TODO: разделите поля на числовые и категориальные.
    # Для чисел можно использовать fillna(), для категорий — pd.get_dummies().
    # Если preparation is None: вычислите медианы по train и сохраните их,
    # а также список полученных столбцов в словарь preparation.
    # Иначе: используйте сохранённые значения без повторного вычисления.
    # Для одинакового набора столбцов пригодится:
    # data.reindex(columns=preparation["columns"], fill_value=0)
    # Верните числовую таблицу и словарь: return data, preparation.
    raise NotImplementedError("Допишите подготовку признаков простыми шагами.")


X_train, preparation = prepare_features(train[features])
X_test, _ = prepare_features(test[features], preparation)
y_train, y_test = train[TARGET], test[TARGET]
assert len(X_train) == len(y_train) and len(X_test) == len(y_test)
assert X_train.index.equals(y_train.index) and X_test.index.equals(y_test.index)
assert list(X_train.columns) == list(X_test.columns)
assert np.isfinite(X_train.to_numpy(dtype=float)).all()
assert np.isfinite(X_test.to_numpy(dtype=float)).all()

## 4. Обучим классификатор

Начать можно с `LogisticRegression` или другой модели с `predict_proba`, чтобы показывать вероятность в приложении. В месте TODO верните выбранную модель. Ниже сравним её с `DummyClassifier` на test.

Для простоты используем фиксированный порог вероятности 0.5: если вероятность не меньше 0.5, получаем класс 1. Подбирать другой порог в этом проекте не требуется.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix,
)


def probability_of_one(model, X):
    # Не полагаемся на произвольный порядок классов в predict_proba.
    classes = np.asarray(model.classes_)
    assert set(classes) == {0, 1}, "Нужны оба класса: 0 и 1."
    column = np.flatnonzero(classes == 1)[0]
    return np.asarray(model.predict_proba(X))[:, column]


def classification_metrics(y_true, probability, threshold):
    prediction = (probability >= threshold).astype(int)
    return {
        "precision": precision_score(y_true, prediction, zero_division=0),
        "recall": recall_score(y_true, prediction, zero_division=0),
        "F1": f1_score(y_true, prediction, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, probability),
    }


from sklearn.linear_model import LogisticRegression


def build_model():
    # TODO: верните выбранный классификатор, например LogisticRegression(max_iter=1000).
    raise NotImplementedError("Выберите модель классификации.")


threshold = 0.5
assert all(set(part[TARGET]) == {0, 1} for part in [train, test])
baseline = DummyClassifier(strategy="prior")
baseline.fit(X_train, y_train)
model = build_model()
model.fit(X_train, y_train)

pd.DataFrame({
    "dummy_baseline": classification_metrics(y_test, probability_of_one(baseline, X_test), threshold),
    "your_model": classification_metrics(y_test, probability_of_one(model, X_test), threshold),
}).T

## 5. Попробуем ваши идеи

Сравните одну и ту же модель без района и улицы, с районом, а затем с районом и улицей. Попробуйте ещё одну идею из задания или предложите свою. Каждое решение обучайте только на train, а метрики запишите на test.

Для нового набора полей измените `features` и заново выполните подготовку и обучение. Сохраните результаты всех вариантов в небольшой таблице. Даже если улучшения нет, расскажите, что заметили.

В нашем упрощённом проекте только train и test. Если по test вы выбираете самый удачный вариант, его метрика уже не является независимой оценкой — просто укажите это в выводах. Для сохранения ниже используется текущий `model` и соответствующие ему `features` и `preparation`.

In [ ]:
# TODO: сравните наборы входов и ещё одну идею, записав test-метрики.
# Например: C у LogisticRegression, class_weight, способ заполнения пропусков.

**TODO:** Какие варианты вы сравнили и что получилось? Кратко опишите наблюдения по таблице метрик.

Ответ:

In [ ]:
# Сохраним текущую обученную модель и те же правила обработки.
# Если после экспериментов возвращаетесь к другой модели, верните также
# её features и preparation, затем заново подготовьте X_test.
best_model = model
best_features = list(features)
best_preparation = preparation
assert best_features and len(best_features) == len(set(best_features))
assert set(best_features).issubset(df.columns)
assert not set(best_features) & forbidden
assert "price_rub" in best_features, "Сохраните общую цену во входах классификатора."

## 6. Запишем результаты и сохраним решение

Ниже получатся test-метрики и матрица ошибок: строки — правильный класс, столбцы — прогноз; порядок классов 0, 1. Используем тот же порог 0.5.

Посмотрите на несколько FP и FN. Какие квартиры модель ошибочно отнесла к классу 1, а какие — к классу 0? Эти примеры помогут рассказать об ограничениях приложения.

In [ ]:
X_final_test, _ = prepare_features(test[best_features], best_preparation)
test_probability = probability_of_one(best_model, X_final_test)
test_prediction = (test_probability >= threshold).astype(int)
test_metrics = classification_metrics(y_test, test_probability, threshold)
display(pd.Series(test_metrics))
pd.DataFrame(
    confusion_matrix(y_test, test_prediction, labels=[0, 1]),
    index=["true_0", "true_1"], columns=["predicted_0", "predicted_1"],
)

In [ ]:
errors = test[["listing_id", TARGET]].copy()
errors["probability"] = test_probability
errors["predicted_class"] = test_prediction
errors.loc[errors[TARGET] != errors["predicted_class"]].head()

**TODO:** Посмотрите на ошибочные прогнозы и объясните на примерах, что означают FP и FN в этой задаче.

Ответ:

In [ ]:
model_path = ROOT / "models" / "class_model.joblib"
model_path.parent.mkdir(exist_ok=True)
joblib.dump({
    "model": best_model, "features": best_features,
    "preparation": best_preparation, "threshold": threshold,
}, model_path)

## 7. Убедимся, что всё работает

В конце нас ждут небольшие проверки: для каждой квартиры должна получиться вероятность от 0 до 1, а после сохранения и загрузки модели ответы должны совпасть. Алгоритм и минимальное качество здесь не заданы — проверки помогут найти техническую ошибку независимо от выбранного вами подхода.

Все ответы должны рассчитываться вашей моделью: тогда эти же шаги будут работать и на новых квартирах.


In [ ]:
assert test_probability.shape == (len(X_final_test),)
assert np.isfinite(test_probability).all()
assert ((0 <= test_probability) & (test_probability <= 1)).all()
assert set(np.unique(test_prediction)).issubset({0, 1})
assert all(np.isfinite(value) for value in test_metrics.values())
assert np.isfinite(threshold) and 0 <= threshold <= 1
assert model_path.is_file()
loaded = joblib.load(model_path)
assert loaded["features"] == best_features
assert loaded["threshold"] == threshold
X_loaded, _ = prepare_features(test[loaded["features"]], loaded["preparation"])
reloaded_probability = probability_of_one(loaded["model"], X_loaded)
assert np.allclose(test_probability, reloaded_probability)
assert np.array_equal(
    test_prediction, (reloaded_probability >= loaded["threshold"]).astype(int),
)
print("Основные проверки пройдены.")

## 8. Расскажите, что у вас получилось

Удалось ли улучшить простой ориентир? Помогли ли район и улица? Какие ошибки оказались интересными? Добавьте таблицу test-метрик всех вариантов и ваши наблюдения. Для всех вариантов используем один порог вероятности 0.5.

Перед подключением к приложению перезапустите ядро и выполните ноутбук сверху вниз. Затем повторите ту же подготовку данных в `app.py`. Пример интерфейса можно найти в отдельном открытом репозитории `streamlit-app` с уже развёрнутым приложением.

Ответ: